In [1]:
import pandas as pd
import numpy as np
import psycopg2
import sqlalchemy as db
from sqlalchemy import create_engine
import yaml

In [2]:
with open('.\\config.yml', 'r') as f:
    config = yaml.safe_load(f)
    config_co = config['mensajeria']
    config_etl = config['ETL_PRO']

# Construct the database URL
url_co = (f"{config_co['drivername']}://{config_co['user']}:{config_co['password']}@{config_co['host']}:"
          f"{config_co['port']}/{config_co['dbname']}")
url_etl = (f"{config_etl['drivername']}://{config_etl['user']}:{config_etl['password']}@{config_etl['host']}:"
           f"{config_etl['port']}/{config_etl['dbname']}")
# Create the SQLAlchemy Engine
mensajeria = create_engine(url_co)
etl_conn = create_engine(url_etl)

## se cargan las respectivas tablas de mensajero, auth_user y ciudad

In [3]:
tabla_mensajero = pd.read_sql_table('clientes_mensajeroaquitoy', mensajeria)
tabla_mensajero.head(10)
tabla_mensajero['activo'].value_counts()

activo
True     49
False     1
Name: count, dtype: int64

In [4]:
tabla_persona = pd.read_sql_table('auth_user', mensajeria)
tabla_persona.head(10)


,id,password,last_login,is_superuser,username,first_name,last_name,email,is_staff,is_active,date_joined
0,318,pbkdf2_sha256$600000$zkyZt2EE1IbKK2BEQIh0qg$cA...,NaT,False,Juan paz,pepito_el_rapido,pepito_el_furioso,rapidos-furiosos@gmail.com,False,True,2023-08-31 15:58:54+00:00
1,322,pbkdf2_sha256$600000$0PRLEKfGpLgULMpmcShBQN$nR...,2024-08-29 14:20:34.103482+00:00,False,Luis Rubio,pepito_el_rapido,pepito_el_furioso,rapidos-furiosos@gmail.com,False,True,2023-09-04 20:48:28+00:00
2,326,pbkdf2_sha256$600000$a4qFxPBCprNQVGCKzoV4zt$vj...,2024-05-23 11:19:34.968180+00:00,False,Alvaro Dominguez,pepito_el_rapido,pepito_el_furioso,rapidos-furiosos@gmail.com,False,True,2023-09-26 22:27:31+00:00
3,327,pbkdf2_sha256$600000$v9Zh6S0AKTvSqRwUnD67gw$8W...,2024-08-08 20:28:39.962222+00:00,False,Luis Castro,pepito_el_rapido,pepito_el_furioso,rapidos-furiosos@gmail.com,False,True,2023-10-05 15:34:53+00:00
4,328,pbkdf2_sha256$600000$kzNKcQLO72lUwSElgSWzvK$u1...,NaT,False,Luis Valenciano,pepito_el_rapido,pepito_el_furioso,rapidos-furiosos@gmail.com,False,False,2023-10-05 15:40:15+00:00
5,329,pbkdf2_sha256$600000$gzhKCjb9Fx2ToaTKMngwCq$LY...,2024-08-31 14:04:08.699380+00:00,False,Carlos Valderrama,pepito_el_rapido,pepito_el_furioso,rapidos-furiosos@gmail.com,False,True,2023-10-05 15:42:11+00:00
6,330,pbkdf2_sha256$600000$ENp314lIw6kPtEUCZ3CHQY$5m...,2024-08-24 02:08:44.987977+00:00,False,Faustino Asprilla,pepito_el_rapido,pepito_el_furioso,rapidos-furiosos@gmail.com,False,True,2023-10-05 15:44:20+00:00
7,331,pbkdf2_sha256$600000$m7EMBZrPCyUlE58k2QAumK$Yh...,2024-08-10 00:14:08.315931+00:00,False,GEOVANNY Hidalgo,pepito_el_rapido,pepito_el_furioso,rapidos-furiosos@gmail.com,False,True,2023-10-05 15:46:48+00:00
8,334,pbkdf2_sha256$600000$SxgdIMy3RTe0lQ1kSmRx9t$h3...,2024-08-29 16:03:38.602757+00:00,False,Juan Solano,pepito_el_rapido,pepito_el_furioso,rapidos-furiosos@gmail.com,False,True,2023-10-05 15:59:41+00:00
9,335,pbkdf2_sha256$600000$q5AdrosAmxx7OQnCwrnhQg$GU...,2024-08-31 19:04:33.746780+00:00,False,Jan Sastre,pepito_el_rapido,pepito_el_furioso,rapidos-furiosos@gmail.com,False,True,2023-10-05 16:10:24+00:00


In [5]:
tabla_ciudad = pd.read_sql_table('ciudad', mensajeria)
tabla_ciudad.head(10)

,ciudad_id,nombre,departamento_id
0,6,BUGA,1
1,5,BOGOTA,2
2,4,PASTO,4
3,3,POPAYAN,3
4,2,PALMIRA,1
5,1,CALI,1
6,7,TULUA,1
7,10,GUACARI,1
8,11,CANDELARIA,1
9,12,CARTAGO,1


In [6]:
dim_mensajero=pd.merge(tabla_mensajero,tabla_persona, left_on='user_id',right_on='id', how='left' )
dim_mensajero['key_mensajero'] = range(1,len(dim_mensajero)+1)
dim_mensajero.drop(columns=['user_id','fecha_entrada','fecha_salida','salario','telefono','activo','is_superuser','email',
                            'token_Firebase','url_foto','id_y','password','last_login','is_staff','is_active','date_joined'], inplace=True)
dim_mensajero=pd.merge(dim_mensajero,tabla_ciudad, left_on='ciudad_operacion_id',right_on='ciudad_id', how='left' )
dim_mensajero.drop(columns=['ciudad_operacion_id','ciudad_id','departamento_id','first_name','last_name'], inplace=True)
dim_mensajero.rename(columns={'nombre':'ciudad_operacion','id_x':'id_operaciones','username':'nombre',}, inplace=True)
#Aun falta hacer limpieza de datos, pero por ahora se deja asi para poder cargar la tabla en el data warehouse
#Hay que limpiar username, reemplazar estos como el nuevo first y last name
dim_mensajero.head(10)

,id_operaciones,nombre,key_mensajero,ciudad_operacion
0,1,mensajero1,1,ACOPI YUMBO
1,42,JPEDROZA,2,CALI
2,48,JULIANVILLANUEVA,3,CALI
3,41,LUISCARDONA,4,CALI
4,13,GEOVANNY Hidalgo,5,PASTO
5,28,JHONMUÑOZ,6,CALI
6,33,JONATANMANZANO,7,TULUA
7,36,LUISGIL,8,CALI
8,8,Luis Castro,9,PALMIRA
9,21,Vladimir Putin,10,CALI


In [7]:
dim_mensajero.to_sql('dim_mensajero', etl_conn, if_exists='replace', index=False) 

50